In [1]:
import cv2
import numpy as np

### define functions

In [17]:
# 显示图像
def cv_show(name,img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# resize图像
def resize(image, width= None, height= None, inter= cv2.INTER_AREA):
    dim= None
    (h,w)= image.shape[:2] # image.shape returns h,w
    if width is None and height is None:
        return image
    if width is None:
        r= height/float(h)
        width= int(r*w)
        dim= (width, height)
    else:
        r= width/(float(w))
        height= int(h*r)
        dim= (width, height)
    resized= cv2.resize(image, dim, interpolation=inter) # but dim should be passed as w,h
    return resized

# 顺时针轮廓点排序
def order_points(pts):
    rect= np.zeros((4, 2), dtype="float32")
    s= pts.sum(axis=1)
    rect[0]= pts[np.argmin(s)] # top-left
    rect[2]= pts[np.argmax(s)] # bottom-right
    diff= np.diff(pts, axis=1)
    rect[1]= pts[np.argmin(diff)] # top-right
    rect[3]= pts[np.argmax(diff)] # bottom-left
    return rect

# 四点透视变换
def four_point_transform(image, pts):
    rect= order_points(pts)
    (tl, tr, br, bl)= rect
    widthA= np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB= np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth= max(int(widthA), int(widthB))
    heightA= np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB= np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight= max(int(heightA), int(heightB))
    
    # 变换后对应坐标位置
    dst= np.array([
        [0, 0],
        [maxWidth-1, 0],
        [maxWidth-1, maxHeight-1],
        [0, maxHeight-1]
    ], dtype="float32")
    
    M= cv2.getPerspectiveTransform(rect, dst) # 计算透视变换矩阵
    warped= cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    
    return warped

### 边缘检测

In [8]:
image= cv2.imread("Receipt.png")

gray= cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray= cv2.GaussianBlur(gray, (5, 5), 0) # 边缘检测前一般先用高斯模糊
edged= cv2.Canny(gray, 75, 200)

### 轮廓检测

In [ ]:
cnts, _ = cv2.findContours(edged.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
cnts= sorted(cnts, key=cv2.contourArea, reverse=True)[:5]

# 计算轮廓的近似多边形
for c in cnts:
    peri= cv2.arcLength(c, True)
    approx= cv2.approxPolyDP(c, 0.02 * peri, True)
    if len(approx) == 4:
        screenCnt= approx
        break

# 对图像进行透视变换
warped= four_point_transform(image, screenCnt.reshape(4, 2))

In [21]:
cv_show("Edged", edged)
cv_show("Original", image) 
cv_show("Warped", warped)